In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    
    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__


## Local MCP server

In [3]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "local_server": {
                "transport": "stdio",
                "command": "python",
                "args": ["resources/2.1_mcp_server.py"],
            }
    }
)

In [4]:
# get tools
tools = await client.get_tools()

# get resources
resources = await client.get_resources("local_server")

# get prompts
prompt = await client.get_prompt("local_server", "prompt")
prompt = prompt[0].content

In [5]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    tools=tools,
    system_prompt=prompt
)

In [6]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Tell me about the langchain-mcp-adapters library")]},
    config=config
)

In [7]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Tell me about the langchain-mcp-adapters library', additional_kwargs={}, response_metadata={}, id='e97106de-5a53-47a9-afff-cfb6c27eb4dd'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 221, 'prompt_tokens': 271, 'total_tokens': 492, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 192, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EGw9S2deoksMW3SabS7BO8q7wJV4h', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a03b8c-f47b-75f2-b3bb-b77a04425b44-0', tool_calls=[{'name': 'search_web', 'args': {'query': 'langchain-mcp-adapters library'}, 'id': 'call_8eWKbqwPWqC82Qt5tFEf4frL', 'type': 'too

In [16]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "-m", "mcp_server_time", "--local-timezone=America/New_York"],
    capture_output=True,
    text=True,
    timeout=3,
    input=""
)
print("STDOUT:", result.stdout)
print("STDERR:", result.stderr)
print("Return code:", result.returncode)

STDOUT: 
STDERR: 
Return code: 0


In [17]:
import os
venv_bin = os.path.dirname(sys.executable)
print(os.listdir(venv_bin))

['jsondiff', 'activate.bat', 'langgraph', 'jupyter-run', 'pyjson5', 'langgraph-verify-graphs', 'jupyter-labextension', 'activate.ps1', 'dotenv', 'mistune', 'python3', 'pybabel', 'jlpm', 'python', 'debugpy', 'distro', 'ipython', 'activate.fish', 'send2trash', 'idna', 'websockets', 'pydoc.bat', 'jupyter-labhub', 'jupyter-server', 'mcp', 'jupyter-dejavu', 'filetype', 'activate_this.py', 'f2py', 'ipython3', 'httpx', 'tb-gcp-uploader', 'langchain-profiles', 'jupyter-nbconvert', 'jupyter-lab', 'jsonschema', 'wsdump', 'mcp-server-time', 'tqdm', 'jupyter-troubleshoot', 'pygmentize', 'jupyter-migrate', 'cffi-gen-src', 'uvicorn', 'activate', 'jupyter-events', 'activate.nu', 'normalizer', 'numpy-config', 'jupyter-trust', 'jsonpointer', 'jsonpatch', 'python-grpc-tools-protoc', 'deactivate.bat', 'python3.12', 'jupyter-builder', 'jupyter-kernelspec', 'jupyter-kernel', 'jupyter', 'debugpy-adapter', 'jupyter-execute', 'watchfiles', 'activate.csh']


## Online MCP

In [18]:
import sys, os

venv_bin = os.path.dirname(sys.executable)

client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": os.path.join(venv_bin, "mcp-server-time"),
            "args": ["--local-timezone=America/New_York"]
        }
    }
)

tools = await client.get_tools()

# client = MultiServerMCPClient(
#     {
#         "time": {
#             "transport": "stdio",
#             "command": sys.executable,
#             "args": [
#                 "run",
#                 "python",
#                 "-m",
#                 "mcp_server_time",
#                 "--local-timezone=America/New_York"
#             ]
#         }
#     }
# )

# tools = await client.get_tools()

In [19]:
agent = create_agent(
    model="gpt-5-nano",
    tools=tools,
)

In [20]:
question = HumanMessage(content="What time is it?")

response = await agent.ainvoke(
    {"messages": [question]}
)

pprint(response)

{'messages': [HumanMessage(content='What time is it?', additional_kwargs={}, response_metadata={}, id='a0cdef7a-3a40-4966-ab91-7d38de2ad921'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 283, 'prompt_tokens': 295, 'total_tokens': 578, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 256, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EGwRUTmKpPKREgragla1NitSPADTY', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a03b9e-03f0-7c02-988f-5488e4bf6cd1-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'America/New_York'}, 'id': 'call_x1GAQniaDUAqykC4P3ku4Xrf', 'type': 'tool_call'}], invalid_tool_calls=[], usa